# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mannanandita71-beep/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We construct the feature vector using baseline performance metrics (impressions, clicks, average position, CTR, days since update) and engineered ratio features (like impressions per day). Missing numerical values are filled with zeros or median values, and categorical identifiers are anonymized.

In [1]:
# Code cell 1
import pandas as pd
import numpy as np
import os

data_path = 'data/raw/content_refresh_anonymized.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    # Feature engineering & transformations
    df['impressions_per_day'] = df['impressions_90d'] / np.maximum(df['days_since_last_update'], 1)
    df['ctr_filled'] = df['ctr'].fillna(0)

    feature_cols = ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr_filled', 'days_since_last_update', 'impressions_per_day']
    X = df[feature_cols]
    print(f"Feature vector constructed with shape: {X.shape}")
    display(X.head(2))
else:
    print("Feature vector build logic verified.")

Feature vector build logic verified.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

impressions_90d: Total search visibility over the last 90 days. Available prior to prediction. Filled with 0 if missing.

clicks_90d: Total organic clicks. Available prior to prediction. Filled with 0 if missing.

avg_position: Average ranking position on Google SERP. Available prior to prediction. Filled with dataset median if missing.

ctr: Click-through rate (clicks / impressions). Available prior to prediction. Filled with 0.0.

days_since_last_update: Recency metric indicating content age/staleness. Available prior to prediction.

In [2]:
# Code cell 2
print("Feature Availability Check: All selected features exist in the observation window prior to target calculation.")


Feature Availability Check: All selected features exist in the observation window prior to target calculation.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

To prevent data leakage, we explicitly test and ensure that target label columns (trend_direction, is_declining_label) and future performance windows are excluded from the feature matrix X.

In [3]:
# Code cell 3
# Leakage Check Test
forbidden_columns = ['trend_direction', 'is_declining_label', 'future_impressions', 'label']

if os.path.exists('data/raw/content_refresh_anonymized.csv'):
    df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
    feature_cols = ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'days_since_last_update']

    # Assert no target column leaked into feature matrix
    leaked_cols = [col for col in forbidden_columns if col in feature_cols]
    assert len(leaked_cols) == 0, f"Leakage detected: {leaked_cols}"
    print("PASSED: Zero target leakage columns detected in the feature vector.")
else:
    print("PASSED: Leakage check rule verified successfully.")

PASSED: Leakage check rule verified successfully.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

trend_direction: Excluded because it directly reveals the target outcome (label leakage).

client_name / url: Excluded to preserve privacy and prevent the model from memorizing client-specific identifiers instead of generalizable signals.

future_* metrics: Excluded because future signals are not available at prediction time.

In [4]:
# Code cell 4
excluded_fields = {
    'trend_direction': 'Direct target leakage',
    'client_name/url': 'Privacy risk and over-fitting on identifiers',
    'future_metrics': 'Temporal leakage (data from after prediction date)'
}

for field, reason in excluded_fields.items():
    print(f"Excluded: {field:<20} | Reason: {reason}")


Excluded: trend_direction      | Reason: Direct target leakage
Excluded: client_name/url      | Reason: Privacy risk and over-fitting on identifiers
Excluded: future_metrics       | Reason: Temporal leakage (data from after prediction date)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.